In [1]:
import numpy as np
import torch
import torch.utils.data as data
import torch.nn as nn
import sys
import warnings
from itertools import product

### NoteBook environment specific (has to be configured on user side)

In [2]:
# import github repo into notebook file directory, might require fixing path
!rm -r /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

# path to the github repo
root_dir = "/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection"
sys.path.append(root_dir)

import utils.config as config
# arg1: path to the cats dataset, arg2: path to the checkpoint folder
config.init("/kaggle/input/datasets/kirillvishnyakov/cats-dataset/data.csv", root_dir + "/checkpoint_dir")

Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 512, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 512 (delta 10), reused 47 (delta 8), pack-reused 455 (from 2)
Receiving objects: 100% (512/512), 178.51 MiB | 38.14 MiB/s, done.
Resolving deltas: 100% (243/243), done.
286886c (HEAD -> main, origin/main, origin/HEAD) refactoring, report checkpoint


### Setup the environment to have access to required repo functions

In [3]:
warnings.filterwarnings("ignore")
from models.lstm_ae import lstm_ae
import dataset
import train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(device)

2.9.0+cu126
cuda


### Random states for reproducibility

In [4]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## tuning helper

In [5]:
def tune_over_grid(hyperparam_grid, train_dataset, val_dataset, epochs = 10):
    results = []
    
    for lookback_window, embedding_dim_ratio, learning_rate in product(
        hyperparam_grid["lookback_window"],
        hyperparam_grid["embedding_dim_ratio"],
        hyperparam_grid["learning_rate"]
    ):
    
        model = lstm_ae(17, lookback_window, embedding_dim_ratio).to(device)
    
        name = f"lb_{lookback_window}_edr_{embedding_dim_ratio}_lr_{learning_rate}"
    
        weights, val_loss, train_loss = train.fit(
            device, model, name, train_dataset, val_dataset, learning_rate, 512, epochs
        )
    
        results.append({
            "val_loss": val_loss,
            "name": name,
            "weights": weights
        })
    results.sort(key=lambda x: x['val_loss'])
    return results

In [6]:
train_dataset = dataset.AE_Dataset(device, 256, start=0, end=800000)
val_dataset = dataset.AE_Dataset(device, 256, scaler=train_dataset.scaler, start=800000, end=1000000, train=False)

## Tune over latent space ratios

In [7]:
hyperparam_grid = \
{ "lookback_window": [256], 
  "embedding_dim_ratio": [0.5, 0.6, 0.75], 
  "learning_rate": [0.001]
}

results = tune_over_grid(hyperparam_grid, train_dataset, val_dataset)
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

LR Scheduler: 781 warmup steps, 15620 total steps
|lb_256_edr_0.5_lr_0.001| train = 0.0502 | test= 0.0631 | LR: 9.93e-04
|lb_256_edr_0.5_lr_0.001| train = 0.0064 | test= 0.0070 | LR: 9.40e-04
|lb_256_edr_0.5_lr_0.001| train = 0.0043 | test= 0.0049 | LR: 8.40e-04
|lb_256_edr_0.5_lr_0.001| train = 0.0036 | test= 0.0039 | LR: 7.04e-04
|lb_256_edr_0.5_lr_0.001| train = 0.0035 | test= 0.0038 | LR: 5.46e-04
|lb_256_edr_0.5_lr_0.001| train = 0.0034 | test= 0.0027 | LR: 3.83e-04
|lb_256_edr_0.5_lr_0.001| train = 0.0030 | test= 0.0024 | LR: 2.34e-04
|lb_256_edr_0.5_lr_0.001| train = 0.0029 | test= 0.0023 | LR: 1.14e-04
|lb_256_edr_0.5_lr_0.001| train = 0.0028 | test= 0.0022 | LR: 3.68e-05
|lb_256_edr_0.5_lr_0.001| train = 0.0028 | test= 0.0022 | LR: 1.00e-05
LR Scheduler: 781 warmup steps, 15620 total steps
|lb_256_edr_0.6_lr_0.001| train = 0.0467 | test= 0.0654 | LR: 9.93e-04
|lb_256_edr_0.6_lr_0.001| train = 0.0060 | test= 0.0058 | LR: 9.40e-04
|lb_256_edr_0.6_lr_0.001| train = 0.0035 | test=

## Test lr - higher ones since training is stable and loss is still decreasing. <br>Choose edr on the lower side even though loss tends to decrease as edr decreases.

In [ ]:
hyperparam_grid = \
{ "lookback_window": [256], 
  "embedding_dim_ratio": [0.55], 
  "learning_rate": [0.0015, 0.002, 0.003]
}

results = tune_over_grid(hyperparam_grid, train_dataset, val_dataset)
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

LR Scheduler: 781 warmup steps, 15620 total steps
|lb_256_edr_0.55_lr_0.0015| train = 0.0396 | test= 0.0437 | LR: 1.49e-03
|lb_256_edr_0.55_lr_0.0015| train = 0.0039 | test= 0.0034 | LR: 1.41e-03
|lb_256_edr_0.55_lr_0.0015| train = 0.0026 | test= 0.0022 | LR: 1.26e-03
|lb_256_edr_0.55_lr_0.0015| train = 0.0024 | test= 0.0019 | LR: 1.06e-03
|lb_256_edr_0.55_lr_0.0015| train = 0.0023 | test= 0.0017 | LR: 8.19e-04
|lb_256_edr_0.55_lr_0.0015| train = 0.0022 | test= 0.0016 | LR: 5.75e-04
|lb_256_edr_0.55_lr_0.0015| train = 0.0022 | test= 0.0015 | LR: 3.51e-04
|lb_256_edr_0.55_lr_0.0015| train = 0.0021 | test= 0.0015 | LR: 1.72e-04
|lb_256_edr_0.55_lr_0.0015| train = 0.0021 | test= 0.0015 | LR: 5.52e-05
|lb_256_edr_0.55_lr_0.0015| train = 0.0021 | test= 0.0015 | LR: 1.50e-05
LR Scheduler: 781 warmup steps, 15620 total steps
|lb_256_edr_0.55_lr_0.002| train = 0.0336 | test= 0.0443 | LR: 1.99e-03
|lb_256_edr_0.55_lr_0.002| train = 0.0044 | test= 0.0033 | LR: 1.88e-03
|lb_256_edr_0.55_lr_0.002|

## Train final model with more epochs - slightly lower lr to account for the cosine decay landscape keeping a higher lr for longer because this time the model is training for 30 epochs

In [7]:
hyperparam_grid = \
{ "lookback_window": [256], 
  "embedding_dim_ratio": [0.55], 
  "learning_rate": [0.00225]
}

results = tune_over_grid(hyperparam_grid, train_dataset, val_dataset, epochs = 15)
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

LR Scheduler: 781 warmup steps, 23430 total steps
|lb_256_edr_0.55_lr_0.00225| train = 0.0324 | test= 0.0257 | LR: 2.24e-03
|lb_256_edr_0.55_lr_0.00225| train = 0.0038 | test= 0.0036 | LR: 2.19e-03
|lb_256_edr_0.55_lr_0.00225| train = 0.0030 | test= 0.0030 | LR: 2.09e-03
|lb_256_edr_0.55_lr_0.00225| train = 0.0026 | test= 0.0020 | LR: 1.94e-03
|lb_256_edr_0.55_lr_0.00225| train = 0.0024 | test= 0.0017 | LR: 1.76e-03
|lb_256_edr_0.55_lr_0.00225| train = 0.0024 | test= 0.0017 | LR: 1.55e-03
|lb_256_edr_0.55_lr_0.00225| train = 0.0022 | test= 0.0016 | LR: 1.32e-03
|lb_256_edr_0.55_lr_0.00225| train = 0.0022 | test= 0.0015 | LR: 1.08e-03
|lb_256_edr_0.55_lr_0.00225| train = 0.0022 | test= 0.0015 | LR: 8.38e-04
|lb_256_edr_0.55_lr_0.00225| train = 0.0021 | test= 0.0015 | LR: 6.15e-04
|lb_256_edr_0.55_lr_0.00225| train = 0.0021 | test= 0.0015 | LR: 4.15e-04
|lb_256_edr_0.55_lr_0.00225| train = 0.0021 | test= 0.0014 | LR: 2.50e-04
|lb_256_edr_0.55_lr_0.00225| train = 0.0021 | test= 0.0014 | L

In [8]:
torch.save(results_best["weights"], 'lllstm_AutoEncoder_weights.pth')